In [ ]:
import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

from catboost import CatBoostRegressor

import optuna

import nbformat


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


from transformers import AutoTokenizer, AutoModel

import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet')
df['price_per_m2'] = df['price_numeric'] / df['area']
df['description'] = df['description'].apply(lambda x: str(x))
df['address'] = df['address'].apply(lambda x: str(x))
df['area'] = df['area'].fillna(0)
df.shape

NameError: name 'pd' is not defined

In [ ]:
df['area'].isna().sum()

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
def get_encoders(df, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor']):
    label_encoders = {}
    for col in columns:
        enc = LabelEncoder()
        enc.fit(df[col])
        label_encoders[col] = enc
    return label_encoders

def apply_encoders(df, encoders, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor']):
    df_cp = df.copy()
    for col in columns:
        df_cp[col] = encoders[col].transform(df_cp[col] )
    return df_cp

def scale(df, columns=['price_numeric','area', 'price_per_m2']):
    scaler = StandardScaler()
    scaler.fit(df[columns])
    df[columns] = scaler.transform(df[columns])
    return scaler

In [ ]:
class RoualtyDataset(Dataset):
    def __init__(self, df, tokenizer):

        self.df = df
        self.tokenizer = tokenizer

        self.num_features = ['area']
        self.cat_features = ['rooms', 'metro', 'title', 'self_floor', 'max_floor']


    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx):

        item = self.df.iloc[idx]
        
        numeric = item[self.num_features]
        category = item[self.cat_features]


        description_text = f"Описание недвижимости: item['description']"
        address_text = f"Описание недвижимости: item['description']"

        description = self.tokenizer(description_text, padding=True, truncation=True, return_tensors='pt')['input_ids'].squeeze(0)
        address = self.tokenizer(address_text, padding=True, truncation=True, return_tensors='pt')['input_ids'].squeeze(0)

            
        return {'num_featues': torch.tensor(numeric, dtype=torch.float32),
                'cat_features':torch.tensor(category, dtype=torch.long) ,
                'description':description,
                'address':address,
                'target':torch.tensor(self.df['price_numeric'].iloc[idx], dtype=torch.float32)
                }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def precompute_rope_frequencies(dim, seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(seq_len)
    freqs = torch.outer(t, freqs).float()
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    
    rotated_x = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated_x.flatten(-2)

class RoPETransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # Feed-Forward
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.SiLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, cos, sin):
        residual = x
        x = self.norm1(x)
        
        batch_size, seq_len, _ = x.shape

        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        cos_input = cos[:seq_len, :].view(1, 1, seq_len, -1)
        sin_input = sin[:seq_len, :].view(1, 1, seq_len, -1)
        
        q = apply_rope(q, cos_input, sin_input)
        k = apply_rope(k, cos_input, sin_input)
        
        attn_output = F.scaled_dot_product_attention(q, k, v)
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)
        
        x = residual + self.out_proj(attn_output)

        x = x + self.ffn(self.norm2(x))
        
        return x

In [ ]:
import torch
import torch.nn as nn

class RoyaltyModel(nn.Module):
    def __init__(self, text_encoder, emb_dim=128, emb_cnt=[13, 349, 37, 78, 78], n_blocks=3):
        super().__init__()
        self.text_encoder = text_encoder
        self.emb_dim = emb_dim
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(cnt, emb_dim) for cnt in emb_cnt
        ])
    
        self.desc_mlp = nn.Sequential(
            nn.Linear(312, 128), nn.SiLU(),
            nn.Linear(128, emb_dim), nn.SiLU()
        )

        self.address_mlp = nn.Sequential(
            nn.Linear(312, 128), nn.SiLU(),
            nn.Linear(128, emb_dim), nn.SiLU()
        )
    
        self.area_mlp = nn.Sequential(
            nn.Linear(1, 64), nn.SiLU(),
            nn.Linear(64, emb_dim)
        )
        
        self.transformers = nn.ModuleList([
            RoPETransformerBlock(embed_dim=emb_dim, num_heads=8, ff_dim=512, dropout=0.1) 
            for _ in range(n_blocks)
        ])

        self.max_seq_len = 2 + 1 + len(emb_cnt) # desc + addr + area + cats = 8
        cos, sin = precompute_rope_frequencies(emb_dim // 8, self.max_seq_len)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        self.head = nn.Sequential(
            nn.Linear(emb_dim, 128), nn.SiLU(),
            nn.Linear(128, 1)
        )

    def forward(self, numeric, category, description, address):
        t_desc = self.text_encoder(description).last_hidden_state[:, 0, :]
        t_addr = self.text_encoder(address).last_hidden_state[:, 0, :]
        
        desc_emb = self.desc_mlp(t_desc).unsqueeze(1) 
        addr_emb = self.address_mlp(t_addr).unsqueeze(1)
        area_emb = self.area_mlp(numeric).unsqueeze(1) 

        cat_embs = []
        for i, emb_layer in enumerate(self.embeddings):
            cat_embs.append(emb_layer(category[:, i]).unsqueeze(1))
        
        cat_embs = torch.cat(cat_embs, dim=1) # (B, 5, emb_dim)

        x = torch.cat([desc_emb, addr_emb, area_emb, cat_embs], dim=1) # (B, 8, emb_dim)

        for block in self.transformers:
            x = block(x, self.rope_cos, self.rope_sin)

        x = x.mean(dim=1)
        return self.head(x)


In [ ]:
cat_columns = ['rooms', 'metro', 'title', 'self_floor', 'max_floor']
encoders = get_encoders(df, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor'])
enc_df = apply_encoders(df, encoders, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor'])
scaler = scale(enc_df, columns=['price_numeric'])

emb_cnt = {col:enc_df[col].nunique()+1 for col in cat_columns}
emb_cnt

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
text_encoder = AutoModel.from_pretrained("cointegrated/rubert-tiny2")

In [ ]:
train, val = train_test_split(enc_df, test_size=0.2, shuffle=True)

train_dataset = RoualtyDataset(train, tokenizer=tokenizer)
val_dataset = RoualtyDataset(val, tokenizer=tokenizer)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)


In [ ]:
model = RoyaltyModel(text_encoder, emb_dim=256, n_blocks=4)

print(f"Total_params = {sum(p.numel() for p in model.parameters()) / 1024**3} GB")

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

epochs = 10

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for batch in train_loop:
        numeric = batch['num_featues'].to(device).float()
        category = batch['cat_features'].to(device).long()
        target = batch['target'].to(device).float()
        description = batch['description'].to(device)
        address = batch['address'].to(device)

        optimizer.zero_grad()
        outputs = model(numeric, category, description, address).squeeze()
        
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in val_loader:
            numeric = batch['num_featues'].to(device).float()
            category = batch['cat_features'].to(device).long()
            target = batch['target'].to(device).float()
            description = batch['description'].to(device)
            address = batch['address'].to(device)

            outputs = model(numeric, category, description, address).squeeze()
            
            val_loss += criterion(outputs, target).item()
            
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    r2 = r2_score(all_targets, all_preds)
    mae = mean_absolute_error(all_targets, all_preds)
    mape = mean_absolute_percentage_error(all_targets, all_preds)

    scheduler.step() 

    print(f"\n[Epoch {epoch+1}] Results:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val MSE:    {avg_val_loss:.4f}")
    print(f"  Val MAE:    {mae:.4f}")
    print(f"  Val R2:     {r2:.4f}")
    print(f"  Val MAPE:   {mape:.2%}")
    print(f"  Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    print("-" * 30)
